# Task 3 - Hyper Optimized LinearSVC (The Final Push)

We have established that Linear classifiers work best on this sparse data.
However, to maximize **Macro F1** specifically, we must address two critical hurdles that cap our score:

1. **Information Dilution:** Standard English words (like 'the', 'is', 'in') pad the documents and dilute the TF-IDF vectors of crucial technical jargon. We will prune them using `stop_words='english'`, and restrict `max_df=0.50` so any word appearing in >50% of documents is deleted.
2. **The Macro F1 Trap:** Macro F1 treats all 40 classes equally. If class 'X' has 10,000 papers and class 'Y' has 50 papers, 'Y' is ignored by standard accuracy models. We must employ `class_weight='balanced'` to physically force the SVC penalty to prioritize minority classes equally.

In [ ]:
# Section 0 - Imports & Config
import os
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC

warnings.filterwarnings('ignore')

CONFIG = {
    'TRAIN_PATH': 'dataset/train.csv',
    'TEST_PATH': 'dataset/test.csv',
    'SUBMISSIONS_DIR': 'submissions',
    'RANDOM_STATE': 42,
    'VAL_SIZE': 0.15,
    # The Hyper-Optimized TF-IDF
    'TFIDF': {
        'max_features': 150000,
        'ngram_range': (1, 3),
        'min_df': 2,
        'max_df': 0.50, # Drop common language
        'stop_words': 'english', # Nuke structure words completely
        'sublinear_tf': True,
    }
}

os.makedirs(CONFIG['SUBMISSIONS_DIR'], exist_ok=True)
np.random.seed(CONFIG['RANDOM_STATE'])
print('Config loaded.')

## Section 1 - Load Data & Split

In [ ]:
train_df = pd.read_csv(CONFIG['TRAIN_PATH'], sep='\\t')
test_df = pd.read_csv(CONFIG['TEST_PATH'], sep='\\t')

train_texts = train_df['abstract'].astype(str)
train_labels = train_df['label_id'].astype(int)
test_texts = test_df['abstract'].astype(str)
test_ids = test_df['id'].astype(int)

# Validation Split
X_train_text, X_val_text, y_train, y_val = train_test_split(
    train_texts, train_labels, test_size=CONFIG['VAL_SIZE'], 
    random_state=CONFIG['RANDOM_STATE'], stratify=train_labels
)

print(f'Train split: {len(X_train_text)} samples')
print(f'Val split:   {len(X_val_text)} samples')

## Section 2 - Aggressive TF-IDF Generation
Pruning stop words and computing trigrams...

In [ ]:
print('Fitting 150k Aggressively Pruned TF-IDF...')
t0 = time.time()
vec = TfidfVectorizer(**CONFIG['TFIDF'])
X_train_vec = vec.fit_transform(X_train_text)
X_val_vec = vec.transform(X_val_text)
X_test_vec = vec.transform(test_texts)
print(f'TF-IDF Done in {time.time()-t0:.2f}s. Shape: {X_train_vec.shape}')

## Section 3 - The Balanced SVC Sweeper
We test the raw accuracy-focused SVC vs the Macro-F1 Balanced SVC.

In [ ]:
models = {
    'LinearSVC (Unbalanced)': LinearSVC(C=1.0, max_iter=2000, random_state=CONFIG['RANDOM_STATE']),
    'LinearSVC (Balanced)':   LinearSVC(C=1.0, max_iter=2000, class_weight='balanced', random_state=CONFIG['RANDOM_STATE'])
}

for name, model in models.items():
    print(f'Training {name}...')
    t0 = time.time()
    model.fit(X_train_vec, y_train)
    preds = model.predict(X_val_vec)
    
    f1 = f1_score(y_val, preds, average='macro')
    acc = accuracy_score(y_val, preds)
    print(f"{name} Val Macro F1: {f1:.4f}  |  Val Accuracy: {acc:.4f}  (Time: {time.time()-t0:.2f}s)")

## Section 4 - Generating Final CSV Submission
We will output the Balanced LinearSVC, as it explicitly optimizes the Kaggle metric (Macro F1).

In [ ]:
print('Retraining Balanced LinearSVC on ALL data for max Kaggle performance...')
final_svc = LinearSVC(C=1.0, max_iter=2000, class_weight='balanced', random_state=CONFIG['RANDOM_STATE'])
X_train_all = vec.transform(train_texts)
final_svc.fit(X_train_all, train_labels)

test_preds = final_svc.predict(X_test_vec)
sub = pd.DataFrame({'id': test_ids, 'label_id': test_preds})
out_file = os.path.join(CONFIG['SUBMISSIONS_DIR'], 'task3_hyper_optimized.csv')
sub.to_csv(out_file, index=False)

print(f'Saved {out_file}. Ready for Kaggle upload!')